In [9]:
import json
import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressionModel

def main():
    print("=" * 60)
    print(" INFERENCE DEMO - DỰ ĐOÁN CẤP ZONE TỪ CỤM (DISAGGREGATION)")
    print("=" * 60)

    # 1. Khởi tạo Spark Session (Dùng tài nguyên local để test)
    spark = SparkSession.builder \
        .appName("TaxiDemandZoneInference") \
        .master("local[*]") \
        .getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

    # 2. Load Model GBT
    model_path = "hdfs:///user/vietanhdata/models/taxi_demand_gbt_model"
    print(f"[1] Đang load mô hình GBT từ: {model_path}...")
    try:
        gbt_model = GBTRegressionModel.load(model_path)
        print(" Load mô hình thành công!")
    except Exception as e:
        print(f" Lỗi khi load mô hình: {e}")
        return

    # 3. Load và xử lý file Tỷ trọng (Ratios JSON)
    ratios_file = "zone_cluster_ratios.json"
    print(f"\n[2] Đang load từ điển tỷ trọng Zone từ: {ratios_file}...")
    try:
        with open(ratios_file, "r") as f:
            ratios_data = json.load(f)
        
        zone_lookup = {}
        for cluster_id_str, zones_dict in ratios_data["ratios"].items():
            for zone_id_str, prop in zones_dict.items():
                zone_lookup[zone_id_str] = (int(cluster_id_str), float(prop))
        
        print(f" Đã nạp thành công {len(zone_lookup)} zones vào bộ nhớ RAM!")
    except Exception as e:
        print(f" Lỗi khi đọc file JSON: {e}")
        return

    # =====================================================================
    # BẮT ĐẦU LUỒNG DỰ ĐOÁN (MÔ PHỎNG REQUEST TỪ NGƯỜI DÙNG)
    # =====================================================================
    
    # Người dùng muốn dự đoán cho Zone 
    target_zone = "250" 
    print(f"\n[3] Nhận request dự đoán cho Zone ID: {target_zone}")
    
    # Kiểm tra xem Zone này có tồn tại không
    if target_zone not in zone_lookup:
        print(f" Zone {target_zone} không tồn tại trong dữ liệu lịch sử!")
        return
        
    # Tra cứu Cluster_ID và Tỷ trọng ngay lập tức
    target_cluster, proportion = zone_lookup[target_zone]
    print(f"    ▶ Vùng {target_zone} thuộc Cụm (Cluster): {target_cluster}")
    print(f"    ▶ Tỷ trọng lịch sử chiếm: {proportion*100:.2f}%")

    # 4. Tạo dữ liệu giả lập (Random Sample) 
    columns = [
        'cluster_id', 'hour', 'day_of_week', 'is_weekend', 'month', 
        'lag_1', 'lag_2', 'lag_3', 'lag_48', 'lag_336'
    ]
    
    sample_data = [(
        target_cluster,                    # Truyền Cluster ID vừa tra cứu được
        int(np.random.randint(0, 24)),     # hour (0 -> 23)
        int(np.random.randint(1, 8)),      # day_of_week (1 -> 7)
        int(np.random.choice([0, 1])),     # is_weekend
        int(np.random.randint(1, 13)),     # month
        float(np.random.randint(10, 500)), # lag_1
        float(np.random.randint(10, 500)), # lag_2
        float(np.random.randint(10, 500)), # lag_3
        float(np.random.randint(10, 500)), # lag_48
        float(np.random.randint(10, 500))  # lag_336
    )]
    
    df_sample = spark.createDataFrame(sample_data, columns)

    # 5. Tiền xử lý (VectorAssembler) & Chạy Inference
    print("\n[4] Đang dự đoán tổng nhu cầu của Cụm qua model GBT...")
    assembler = VectorAssembler(inputCols=columns, outputCol="features")
    df_features = assembler.transform(df_sample)

    predictions = gbt_model.transform(df_features)
    predicted_cluster_demand = predictions.select("prediction").first()[0]
    
    # 6. BƯỚC QUAN TRỌNG: Phân rã (Disaggregation) xuống cấp Zone
    predicted_zone_demand = predicted_cluster_demand * proportion

    # 7. In kết quả cuối cùng
    print(f"\n============================================================")
    print(f" KẾT QUẢ DỰ ĐOÁN CUỐI CÙNG")
    print(f"============================================================")
    print(f" Dự đoán tổng Cụm {target_cluster:<2}   : {predicted_cluster_demand:.2f} chuyến")
    print(f" Tỷ trọng Zone {target_zone:<4}     : {proportion*100:.2f}%")
    print(f" DỰ ĐOÁN CHO ZONE {target_zone:<4} : {predicted_zone_demand:.0f} chuyến taxi")
    print(f"============================================================")

    spark.stop()

if __name__ == "__main__":
    main()

 INFERENCE DEMO - DỰ ĐOÁN CẤP ZONE TỪ CỤM (DISAGGREGATION)
[1] Đang load mô hình GBT từ: hdfs:///user/vietanhdata/models/taxi_demand_gbt_model...
 Load mô hình thành công!

[2] Đang load từ điển tỷ trọng Zone từ: zone_cluster_ratios.json...
 Đã nạp thành công 263 zones vào bộ nhớ RAM!

[3] Nhận request dự đoán cho Zone ID: 250
    ▶ Vùng 250 thuộc Cụm (Cluster): 27
    ▶ Tỷ trọng lịch sử chiếm: 1.81%

[4] Đang dự đoán tổng nhu cầu của Cụm qua model GBT...



 KẾT QUẢ DỰ ĐOÁN CUỐI CÙNG
 Dự đoán tổng Cụm 27   : 174.03 chuyến
 Tỷ trọng Zone 250      : 1.81%
 DỰ ĐOÁN CHO ZONE 250  : 3 chuyến taxi
